<a id="top"></a>

# Demo: one wired round-trip, end to end

```yaml
title: "Demo: one wired round-trip, end to end"
keywords: round-trip, dispatch, ToolMessage, tool_call_id, wire a tool, name description schema, langchain, bind_tools, teacher demo
estimated duration: 12
```

> **Lesson:** L07. Teacher demo — Demo 2 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L07/demos_or_activities.md). Makes **live** Claude calls — set `ANTHROPIC_API_KEY` before running. Clear outputs before committing.
>
> **Model-agnostic by design.** The model is a LangChain `ChatAnthropic`; the tool is a plain typed function bound with `bind_tools`. The tool call arrives as `AIMessage.tool_calls`, and we hand the result back as a `ToolMessage`. Swap `ChatAnthropic` for `ChatOpenAI` and the round-trip code is unchanged. The key loads through the config seam (`require_anthropic_key`); we never hard-code it.
>
> The point to land: wiring a tool is **name → describe → bind → dispatch → result → continue**. The application validated and ran the function; the model only proposed.

## Contents

- [1. Setup](#1-setup)
- [2. First turn — the model proposes a call](#2-first-turn--the-model-proposes-a-call)
- [3. Dispatch — the application runs the function](#3-dispatch--the-application-runs-the-function)
- [4. Continue — hand the result back as a ToolMessage](#4-continue--hand-the-result-back-as-a-toolmessage)
- [5. The tool closed the accuracy gap](#5-the-tool-closed-the-accuracy-gap)
- [6. Takeaways](#6-takeaways)

## 1. Setup

Reuse Demo 1's model, tool, and prompt so the technique completes on something familiar. Live cells are gated on `LIVE` so a keyless run still executes top to bottom.

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, ToolMessage

from fluffy_potato_curriculum.common.config import get_settings, require_anthropic_key

SONNET = "claude-sonnet-4-6"
LIVE = bool(get_settings().anthropic_api_key)


# The ONE tool that carries all four demos: a deterministic calculator.
# L07 is deliberately single-tool, single-round-trip (multi-tool is L08, the
# agent loop is L10). Resist adding a second tool.
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression and return the exact result as a string.

    Use this whenever the user asks for the result of a calculation. Restricted to
    digits and the operators + - * / ( ) . and whitespace so a hallucinated
    expression cannot run arbitrary code. Raises ValueError on anything else.
    """
    allowed = set("0123456789+-*/(). ")
    if not expression or set(expression) - allowed:
        raise ValueError(f"refusing to evaluate non-arithmetic expression: {expression!r}")
    # eval is safe here ONLY because the character whitelist above blocks names,
    # attributes, and calls. Never eval untrusted input without such a guard.
    return str(eval(expression))


# The tool definition is DERIVED from the typed function + docstring by bind_tools —
# the model never sees the Python function itself, only the derived definition.
TOOLS = [calculator]

PROMPT = "What is 18,374 multiplied by 92,431?"

if LIVE:
    model = ChatAnthropic(model=SONNET, api_key=require_anthropic_key(), max_tokens=400)
    bound = model.bind_tools(TOOLS)

print("setup ready (LIVE =", LIVE, ")")

[↑ Back to top](#top)

## 2. First turn — the model proposes a call

The same first turn as Demo 1: the model returns a reply whose `.tool_calls` carries one request. Last time we stopped here.

In [ ]:
if LIVE:
    messages = [HumanMessage(PROMPT)]
    first = bound.invoke(messages)
    call = first.tool_calls[0]  # a dict: {"name", "args", "id"}
    print("model proposed:", call["name"], call["args"], "id=", call["id"])
else:
    print("offline: set ANTHROPIC_API_KEY to run this live cell")

[↑ Back to top](#top)

## 3. Dispatch — the application runs the function

Now the step Demo 1 skipped: match the tool name, pull the `expression` argument, run the **real** function, and capture the result. *This* number is computed, not generated.

In [ ]:
if LIVE:
    assert call["name"] == "calculator"  # dispatch on the name the model echoed back
    args = call["args"]  # already a parsed dict, not a string blob
    result = calculator(**args)  # the application runs the tool
    print("application matched tool :", call["name"])
    print("application extracted args:", args)
    print("calculator(...) returned  :", result, "  <- a REAL computed number")
else:
    print("offline: set ANTHROPIC_API_KEY to run this live cell")

[↑ Back to top](#top)

## 4. Continue — hand the result back as a ToolMessage

Record the assistant's turn (append the `AIMessage` itself — don't rebuild it), then append a **`ToolMessage`** carrying the function's output and the **same** `tool_call_id`. Send the *full* conversation back. The model returns a final answer that uses the real number.

*Under the hood:* on the Anthropic wire that `ToolMessage` becomes a `tool_result` block in a user-role message; LangChain normalizes it to its own message role so the same code runs on any provider.

In [ ]:
if LIVE:
    # 1) record the assistant's tool-calling turn (the AIMessage object itself),
    # 2) append the result as a ToolMessage tagged with the same call id.
    messages.append(first)
    messages.append(ToolMessage(content=result, tool_call_id=call["id"]))

    final = bound.invoke(messages)  # the definition rides along AGAIN — the model is stateless
    print("final answer:\n", final.text)
else:
    print("offline: set ANTHROPIC_API_KEY to run this live cell")

[↑ Back to top](#top)

## 5. The tool closed the accuracy gap

Put Demo 1's tool-free guess next to the calculator's real answer — same model, same question.

In [ ]:
if LIVE:
    true_value = 18374 * 92431
    print("ground truth (Python)      :", true_value)
    print("calculator tool result     :", result)
    print("model's final answer used it and is now correct.")
else:
    print("offline: run with a key to compare the tool-free guess vs the real answer")

[↑ Back to top](#top)

## 6. Takeaways

- The five mechanical steps, in one pass: **name** the function, **describe** it (its docstring), **bind** it, **dispatch** on the returned call, run the **result**, **continue** the conversation. That is the whole Objective-1 skill.
- The **application** validated the arguments and ran the function. The model proposed; the application disposed.
- The result went back as a **`ToolMessage`** tagged with the same `tool_call_id` — its own message role, not part of the assistant turn. This sets up [Demo 3](L0706_lecture.ipynb)'s message-by-message trace.
- The tool definition rode along on the *second* call too: the model is **stateless**, so the schema is in every prompt.

[↑ Back to top](#top)